In [1]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

ledger = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv = pd.read_csv(BASE_PATH + 'inventory_master.csv')
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')

print('ledger', ledger.shape)
print('inv', inv.shape)
print('po_h', po_h.shape)
print('po_l', po_l.shape)
print('so_h', so_h.shape)
print('so_l', so_l.shape)

adj = ledger[ledger['movement_type'] == 'ADJUSTMENT']
print('adjustment rows', len(adj))
print('adjustment units', adj['quantity'].sum())

Mounted at /content/drive
ledger (237230, 9)
inv (180, 8)
po_h (24000, 10)
po_l (155495, 9)
so_h (20000, 11)
so_l (130402, 9)
adjustment rows 2454
adjustment units 66990


In [2]:
led = ledger.copy()
led['row_order'] = range(len(led))
led = led.merge(inv[['product_id', 'branch_id', 'opening_stock']], on=['product_id', 'branch_id'], how='left')
led = led.sort_values(['product_id', 'branch_id', 'movement_date', 'row_order'], kind='stable')

led['previous_balance'] = led.groupby(['product_id', 'branch_id'])['running_balance'].shift(1)
led['previous_balance'] = led['previous_balance'].fillna(led['opening_stock'])
led['balance_change'] = led['running_balance'] - led['previous_balance']

in_rows = led[led['movement_type'] == 'IN']
out_rows = led[led['movement_type'] == 'OUT']
adj_rows = led[led['movement_type'] == 'ADJUSTMENT'].copy()

print('purchase rows', len(in_rows), 'balance rose by the quantity', (in_rows['balance_change'] == in_rows['quantity']).sum())
print('sales rows', len(out_rows), 'balance fell by the quantity', (out_rows['balance_change'] == -out_rows['quantity']).sum())

adj_rows['direction'] = 'unreadable'
adj_rows.loc[adj_rows['balance_change'] == adj_rows['quantity'], 'direction'] = 'added'
adj_rows.loc[adj_rows['balance_change'] == -adj_rows['quantity'], 'direction'] = 'subtracted'

print(adj_rows.groupby('direction')['quantity'].agg(['count', 'sum']))

purchase rows 127611 balance rose by the quantity 127611
sales rows 107165 balance fell by the quantity 107165
            count    sum
direction               
added        1227  33526
subtracted   1227  33464


In [3]:
handled = led[led['movement_type'].isin(['IN', 'OUT'])]['quantity'].sum()
subtracted_units = adj_rows[adj_rows['direction'] == 'subtracted']['quantity'].sum()
added_units = adj_rows[adj_rows['direction'] == 'added']['quantity'].sum()

print('units handled', handled)
print('units subtracted', subtracted_units)
print('units added', added_units)
print('net adjustment', added_units - subtracted_units)
print('gross rate percent', round(subtracted_units / handled * 100, 4))

by_branch = pd.DataFrame({
    'handled': led[led['movement_type'].isin(['IN', 'OUT'])].groupby('branch_id')['quantity'].sum(),
    'subtracted': adj_rows[adj_rows['direction'] == 'subtracted'].groupby('branch_id')['quantity'].sum(),
    'added': adj_rows[adj_rows['direction'] == 'added'].groupby('branch_id')['quantity'].sum()
})
by_branch['net'] = by_branch['added'] - by_branch['subtracted']
by_branch['gross_rate_percent'] = (by_branch['subtracted'] / by_branch['handled'] * 100).round(4)
print(by_branch)

units handled 21522627
units subtracted 33464
units added 33526
net adjustment 62
gross rate percent 0.1555
           handled  subtracted  added   net  gross_rate_percent
branch_id                                                      
AHM001     3539666        5190   4992  -198              0.1466
CHN001     3692804        5980   5334  -646              0.1619
DEL001     3622464        5431   5303  -128              0.1499
HYD001     3548234        5730   5920   190              0.1615
KOL001     3538563        6200   5632  -568              0.1752
PUN001     3580896        4933   6345  1412              0.1378


In [4]:
from scipy.stats import chi2_contingency

handled_rows = led[led['movement_type'].isin(['IN', 'OUT'])].copy()
subtracted_rows = adj_rows[adj_rows['direction'] == 'subtracted'].copy()

handled_rows['year'] = pd.to_datetime(handled_rows['movement_date']).dt.year
subtracted_rows['year'] = pd.to_datetime(subtracted_rows['movement_date']).dt.year

def compare_places(column):
    table = pd.DataFrame({
        'handled_rows': handled_rows.groupby(column).size(),
        'subtracted_rows': subtracted_rows.groupby(column).size()
    }).fillna(0)
    table['share_percent'] = (table['subtracted_rows'] / table['handled_rows'] * 100).round(4)
    p_value = chi2_contingency(table[['subtracted_rows', 'handled_rows']])[1]
    return table, p_value

by_branch_rows, p_branch = compare_places('branch_id')
print(by_branch_rows)
print('branch p-value', round(p_branch, 4))

by_year_rows, p_year = compare_places('year')
print(by_year_rows)
print('year p-value', round(p_year, 4))

by_product_rows, p_product = compare_places('product_id')
print('product lowest share', by_product_rows['share_percent'].idxmin(), by_product_rows['share_percent'].min())
print('product highest share', by_product_rows['share_percent'].idxmax(), by_product_rows['share_percent'].max())
print('product p-value', round(p_product, 4))

           handled_rows  subtracted_rows  share_percent
branch_id                                              
AHM001            39525              196         0.4959
CHN001            43600              226         0.5183
DEL001            37575              202         0.5376
HYD001            35877              207         0.5770
KOL001            40848              218         0.5337
PUN001            37351              178         0.4766
branch p-value 0.5024
      handled_rows  subtracted_rows  share_percent
year                                              
2019         37989            197.0         0.5186
2020         39857            198.0         0.4968
2021         39294            207.0         0.5268
2022         38571            219.0         0.5678
2023         39352            207.0         0.5260
2024         38247            199.0         0.5203
2025          1466              0.0         0.0000
year p-value 0.1377
product lowest share P018 0.3698
product highest sh

In [5]:
adj_rows['year'] = pd.to_datetime(adj_rows['movement_date']).dt.year
print(adj_rows.groupby(['year', 'direction']).size().unstack(fill_value=0))
print('last adjustment date', adj_rows['movement_date'].max())

years_to_2024 = by_year_rows.loc[2019:2024]
p_years_to_2024 = chi2_contingency(years_to_2024[['subtracted_rows', 'handled_rows']])[1]
print('year p-value, 2019 to 2024 only', round(p_years_to_2024, 4))

direction  added  subtracted
year                        
2019         187         197
2020         193         198
2021         213         207
2022         221         219
2023         214         207
2024         199         199
last adjustment date 2024-12-29
year p-value, 2019 to 2024 only 0.8518


In [6]:
purchase_ledger = ledger[ledger['reference_type'] == 'PO'].merge(po_h[['po_id', 'po_status']], left_on='reference_id', right_on='po_id', how='left')
sales_ledger = ledger[ledger['reference_type'] == 'SO'].merge(so_h[['so_id', 'order_status']], left_on='reference_id', right_on='so_id', how='left')

purchase_lines_status = po_l.merge(po_h[['po_id', 'po_status']], on='po_id', how='left')
sales_lines_status = so_l.merge(so_h[['so_id', 'order_status']], on='so_id', how='left')

units_on_received_lines = purchase_lines_status[purchase_lines_status['po_status'] == 'Received']['quantity'].sum()
units_on_delivered_lines = sales_lines_status[sales_lines_status['order_status'] == 'Delivered']['quantity'].sum()

ledger_units_received = purchase_ledger[purchase_ledger['po_status'] == 'Received']['quantity'].sum()
ledger_units_delivered = sales_ledger[sales_ledger['order_status'] == 'Delivered']['quantity'].sum()

ledger_units_cancelled_in = purchase_ledger[purchase_ledger['po_status'] == 'Cancelled']['quantity'].sum()
ledger_units_cancelled_out = sales_ledger[sales_ledger['order_status'] == 'Cancelled']['quantity'].sum()

print('units on Received purchase lines', units_on_received_lines)
print('ledger units from Received purchase orders', ledger_units_received)
print('purchase units missing from the ledger', units_on_received_lines - ledger_units_received)
print('percent missing', round((units_on_received_lines - ledger_units_received) / units_on_received_lines * 100, 2))

print('units on Delivered sales lines', units_on_delivered_lines)
print('ledger units from Delivered sales orders', ledger_units_delivered)
print('sales units missing from the ledger', units_on_delivered_lines - ledger_units_delivered)
print('percent missing', round((units_on_delivered_lines - ledger_units_delivered) / units_on_delivered_lines * 100, 2))

print('ledger units on cancelled purchase orders', ledger_units_cancelled_in)
print('ledger units on cancelled sales orders', ledger_units_cancelled_out)

units on Received purchase lines 22387123
ledger units from Received purchase orders 20149239
purchase units missing from the ledger 2237884
percent missing 10.0
units on Delivered sales lines 1234939
ledger units from Delivered sales orders 1111598
sales units missing from the ledger 123341
percent missing 9.99
ledger units on cancelled purchase orders 248556
ledger units on cancelled sales orders 13234


In [7]:
tables_used = {
    'stock ledger': (ledger, ['product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id', 'running_balance'], ['quantity', 'running_balance']),
    'inventory master': (inv, ['product_id', 'branch_id', 'opening_stock'], ['opening_stock']),
    'purchase orders header': (po_h, ['po_id', 'po_status'], []),
    'sales orders header': (so_h, ['so_id', 'order_status'], []),
    'purchase order lines': (po_l, ['po_id', 'product_id', 'quantity'], ['quantity']),
    'sales order lines': (so_l, ['so_id', 'product_id', 'quantity'], ['quantity'])
}

for table_name, (table, columns, number_columns) in tables_used.items():
    print(table_name, 'rows', len(table))
    for column in columns:
        blanks = table[column].isna().sum() + (table[column].astype(str).str.strip() == '').sum()
        print('  ', column, 'blank', blanks)
    for column in number_columns:
        print('  ', column, 'zero', (table[column] == 0).sum(), 'negative', (table[column] < 0).sum())

stock ledger rows 237230
   product_id blank 0
   branch_id blank 0
   movement_type blank 0
   movement_date blank 0
   quantity blank 0
   reference_type blank 0
   reference_id blank 0
   running_balance blank 0
   quantity zero 0 negative 0
   running_balance zero 0 negative 0
inventory master rows 180
   product_id blank 0
   branch_id blank 0
   opening_stock blank 0
   opening_stock zero 0 negative 0
purchase orders header rows 24000
   po_id blank 0
   po_status blank 0
sales orders header rows 20000
   so_id blank 0
   order_status blank 0
purchase order lines rows 155495
   po_id blank 0
   product_id blank 0
   quantity blank 0
   quantity zero 0 negative 0
sales order lines rows 130402
   so_id blank 0
   product_id blank 0
   quantity blank 0
   quantity zero 0 negative 0


In [8]:
print('purchase order statuses', po_h['po_status'].value_counts().to_dict())
print('sales order statuses', so_h['order_status'].value_counts().to_dict())

print('purchase ledger rows with no order status', purchase_ledger['po_status'].isna().sum())
print('sales ledger rows with no order status', sales_ledger['order_status'].isna().sum())

adjustments_per_pair = adj_rows.groupby(['product_id', 'branch_id']).size()
print('pairs with at least one adjustment', len(adjustments_per_pair), 'of', len(inv))
print('fewest adjustment rows in a pair', adjustments_per_pair.min(), 'most', adjustments_per_pair.max())

directions_per_pair = adj_rows.groupby(['product_id', 'branch_id'])['direction'].nunique()
print('pairs with adjustments in one direction only', (directions_per_pair == 1).sum())

ledger_pairs = set(zip(ledger['product_id'], ledger['branch_id']))
inventory_pairs = set(zip(inv['product_id'], inv['branch_id']))
print('ledger pairs not in the inventory master', len(ledger_pairs - inventory_pairs))

purchase order statuses {'Received': 21630, 'Cancelled': 2370}
sales order statuses {'Delivered': 18033, 'Cancelled': 1967}
purchase ledger rows with no order status 0
sales ledger rows with no order status 0
pairs with at least one adjustment 180 of 180
fewest adjustment rows in a pair 5 most 25
pairs with adjustments in one direction only 0
ledger pairs not in the inventory master 0


In [9]:
print('ledger rows', len(ledger), 'distinct movement ids', ledger['movement_id'].nunique())
print('purchase orders', len(po_h), 'distinct order ids', po_h['po_id'].nunique())
print('sales orders', len(so_h), 'distinct order ids', so_h['so_id'].nunique())
print('inventory master repeated pairs', inv.duplicated(['product_id', 'branch_id']).sum())

same_columns = ['product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id']
identical_adjustments = adj_rows.duplicated(subset=same_columns, keep=False)
print('adjustment rows identical to another adjustment row', identical_adjustments.sum())

ledger rows 237230 distinct movement ids 237230
purchase orders 24000 distinct order ids 24000
sales orders 20000 distinct order ids 20000
inventory master repeated pairs 0
adjustment rows identical to another adjustment row 0


In [10]:
ledger_in_units = ledger[ledger['movement_type'] == 'IN']['quantity'].sum()
ledger_out_units = ledger[ledger['movement_type'] == 'OUT']['quantity'].sum()
print('units in', ledger_in_units, 'units out', ledger_out_units, 'units handled', ledger_in_units + ledger_out_units)

print('added plus subtracted', added_units + subtracted_units, 'all adjustment units', adj['quantity'].sum())
print('net adjustment', added_units - subtracted_units)

last_balance_total = led.groupby(['product_id', 'branch_id']).tail(1)['running_balance'].sum()
rebuilt_total = inv['opening_stock'].sum() + ledger_in_units - ledger_out_units + (added_units - subtracted_units)
print('opening stock total', inv['opening_stock'].sum())
print('last balances added up', last_balance_total, 'opening plus in minus out plus net', rebuilt_total)

print('branch tables: handled', by_branch['handled'].sum(), 'subtracted', by_branch['subtracted'].sum(), 'added', by_branch['added'].sum())

records_gap = (units_on_received_lines - ledger_units_received) - (units_on_delivered_lines - ledger_units_delivered) - ledger_units_cancelled_in + ledger_units_cancelled_out
print('records gap', records_gap)

units in 20397795 units out 1124832 units handled 21522627
added plus subtracted 66990 all adjustment units 66990
net adjustment 62
opening stock total 32969
last balances added up 19305994 opening plus in minus out plus net 19305994
branch tables: handled 21522627 subtracted 33464 added 33526
records gap 1879221


In [11]:
from collections import defaultdict

opening_by_pair = {}
for row in inv.itertuples(index=False):
    opening_by_pair[(row.product_id, row.branch_id)] = row.opening_stock

rows_by_pair = defaultdict(list)
for position, row in enumerate(ledger.itertuples(index=False)):
    rows_by_pair[(row.product_id, row.branch_id)].append((row.movement_date, position, row.movement_type, row.quantity, row.running_balance))

loop_added = 0
loop_subtracted = 0
loop_handled = 0
loop_unreadable = 0
loop_subtracted_by_branch = defaultdict(int)

for pair, pair_rows in rows_by_pair.items():
    pair_rows.sort()
    previous_balance = opening_by_pair[pair]
    for movement_date, position, movement_type, quantity, running_balance in pair_rows:
        if movement_type == 'ADJUSTMENT':
            if running_balance - previous_balance == quantity:
                loop_added += quantity
            elif previous_balance - running_balance == quantity:
                loop_subtracted += quantity
                loop_subtracted_by_branch[pair[1]] += quantity
            else:
                loop_unreadable += 1
        else:
            loop_handled += quantity
        previous_balance = running_balance

print('loop added', loop_added, 'loop subtracted', loop_subtracted)
print('loop units handled', loop_handled)
print('loop unreadable adjustments', loop_unreadable)
print('loop gross rate percent', round(loop_subtracted / loop_handled * 100, 4))
print('subtracted units by branch equal the table', all(loop_subtracted_by_branch[branch] == by_branch.loc[branch, 'subtracted'] for branch in by_branch.index))

loop added 33526 loop subtracted 33464
loop units handled 21522627
loop unreadable adjustments 0
loop gross rate percent 0.1555
subtracted units by branch equal the table True


In [12]:
print(adj_rows.groupby('direction')['quantity'].describe().round(2))

pair_losses = pd.DataFrame({
    'subtracted': adj_rows[adj_rows['direction'] == 'subtracted'].groupby(['product_id', 'branch_id'])['quantity'].sum(),
    'handled': led[led['movement_type'].isin(['IN', 'OUT'])].groupby(['product_id', 'branch_id'])['quantity'].sum()
}).fillna(0)
pair_losses['rate_percent'] = pair_losses['subtracted'] / pair_losses['handled'] * 100

print('pairs', len(pair_losses))
print(pair_losses['rate_percent'].describe().round(4))

first_quarter = pair_losses['rate_percent'].quantile(0.25)
third_quarter = pair_losses['rate_percent'].quantile(0.75)
spread = third_quarter - first_quarter
lower_fence = first_quarter - 1.5 * spread
upper_fence = third_quarter + 1.5 * spread
print('normal range', round(lower_fence, 4), 'to', round(upper_fence, 4))
print('pairs below the range', (pair_losses['rate_percent'] < lower_fence).sum())
print('pairs above the range', (pair_losses['rate_percent'] > upper_fence).sum())

print('five highest pairs')
print(pair_losses.sort_values('rate_percent', ascending=False).head(5).round(4))
print('five lowest pairs')
print(pair_losses.sort_values('rate_percent').head(5).round(4))

             count   mean    std  min   25%   50%   75%   max
direction                                                    
added       1227.0  27.32  13.25  5.0  16.0  27.0  38.5  50.0
subtracted  1227.0  27.27  13.25  5.0  16.0  27.0  38.0  50.0
pairs 180
count    180.0000
mean       0.1556
std        0.0635
min        0.0335
25%        0.1064
50%        0.1482
75%        0.2033
max        0.3092
Name: rate_percent, dtype: float64
normal range -0.039 to 0.3486
pairs below the range 0
pairs above the range 0
five highest pairs
                      subtracted  handled  rate_percent
product_id branch_id                                   
P013       KOL001            370   119661        0.3092
P026       AHM001            359   116712        0.3076
P025       CHN001            386   125550        0.3074
P028       CHN001            356   118135        0.3014
P013       PUN001            339   114830        0.2952
five lowest pairs
                      subtracted  handled  rate_percent


In [13]:
pair_rows = pd.DataFrame({
    'handled_rows': handled_rows.groupby(['product_id', 'branch_id']).size(),
    'subtracted_rows': subtracted_rows.groupby(['product_id', 'branch_id']).size()
}).fillna(0)

print('pairs', len(pair_rows))
print('fewest subtraction rows in a pair', int(pair_rows['subtracted_rows'].min()), 'most', int(pair_rows['subtracted_rows'].max()))

expected_subtracted = pair_rows['handled_rows'] / pair_rows['handled_rows'].sum() * pair_rows['subtracted_rows'].sum()
print('pairs expected to have fewer than 5 subtraction rows', (expected_subtracted < 5).sum())

p_value_pairs = chi2_contingency(pair_rows[['subtracted_rows', 'handled_rows']])[1]
print('pair p-value', round(p_value_pairs, 4))

pairs 180
fewest subtraction rows in a pair 2 most 14
pairs expected to have fewer than 5 subtraction rows 0
pair p-value 0.778


In [14]:
ledger_dates = pd.to_datetime(ledger['movement_date'])
adjustment_dates = pd.to_datetime(adj_rows['movement_date'])
print('first and last ledger date', ledger_dates.min().date(), ledger_dates.max().date())
print('first and last adjustment date', adjustment_dates.min().date(), adjustment_dates.max().date())

subtraction_months = pd.to_datetime(subtracted_rows['movement_date']).dt.to_period('M')
months_in_range = pd.period_range(adjustment_dates.min().to_period('M'), adjustment_dates.max().to_period('M'), freq='M')
subtraction_rows_per_month = subtraction_months.value_counts().reindex(months_in_range, fill_value=0)

print('months from the first to the last adjustment', len(months_in_range))
print('months with no subtraction row', (subtraction_rows_per_month == 0).sum())
print('fewest subtraction rows in a month', subtraction_rows_per_month.min(), 'most', subtraction_rows_per_month.max())

first and last ledger date 2019-01-01 2025-01-28
first and last adjustment date 2019-01-01 2024-12-29
months from the first to the last adjustment 72
months with no subtraction row 0
fewest subtraction rows in a month 8 most 28


In [15]:
import csv

with open(BASE_PATH + 'stock_ledger.csv', newline='') as ledger_file:
    raw_ledger = list(csv.DictReader(ledger_file))
with open(BASE_PATH + 'inventory_master.csv', newline='') as inventory_file:
    raw_inventory = list(csv.DictReader(inventory_file))

def raw_opening(product, branch):
    for row in raw_inventory:
        if row['product_id'] == product and row['branch_id'] == branch:
            return int(row['opening_stock'])

def raw_pair_rows(product, branch):
    rows = [(row['movement_date'], position, row) for position, row in enumerate(raw_ledger) if row['product_id'] == product and row['branch_id'] == branch]
    rows.sort(key=lambda item: (item[0], item[1]))
    return [row for movement_date, position, row in rows]

def check_adjustment_row(direction_in_table):
    movement_id = adj_rows[adj_rows['direction'] == direction_in_table]['movement_id'].iloc[0]
    target = [row for row in raw_ledger if row['movement_id'] == movement_id][0]
    pair_rows = raw_pair_rows(target['product_id'], target['branch_id'])
    position = [row['movement_id'] for row in pair_rows].index(movement_id)
    previous_balance = raw_opening(target['product_id'], target['branch_id']) if position == 0 else int(pair_rows[position - 1]['running_balance'])
    new_balance = int(target['running_balance'])
    quantity = int(target['quantity'])
    by_hand = 'added' if new_balance - previous_balance == quantity else 'subtracted' if previous_balance - new_balance == quantity else 'unreadable'
    print(movement_id, target['product_id'], target['branch_id'], 'quantity', quantity, 'balance before', previous_balance, 'balance after', new_balance)
    print('  by hand', by_hand, 'table', direction_in_table, 'agree', by_hand == direction_in_table)

check_adjustment_row('subtracted')
check_adjustment_row('added')

pair_rows = raw_pair_rows('P019', 'KOL001')
balance = raw_opening('P019', 'KOL001')
units_in = units_out = hand_added = hand_subtracted = 0
for row in pair_rows:
    quantity = int(row['quantity'])
    new_balance = int(row['running_balance'])
    if row['movement_type'] == 'IN':
        units_in += quantity
    elif row['movement_type'] == 'OUT':
        units_out += quantity
    elif new_balance - balance == quantity:
        hand_added += quantity
    else:
        hand_subtracted += quantity
    balance = new_balance

print('P019 KOL001 by hand: opening', raw_opening('P019', 'KOL001'), 'in', units_in, 'out', units_out)
print('  added', hand_added, 'subtracted', hand_subtracted, 'net', hand_added - hand_subtracted)
print('  opening plus in minus out plus net', raw_opening('P019', 'KOL001') + units_in - units_out + hand_added - hand_subtracted, 'last balance', balance)

table_pair = adj_rows[(adj_rows['product_id'] == 'P019') & (adj_rows['branch_id'] == 'KOL001')]
print('  table added', table_pair[table_pair['direction'] == 'added']['quantity'].sum(), 'table subtracted', table_pair[table_pair['direction'] == 'subtracted']['quantity'].sum())
print('  table last balance', led[(led['product_id'] == 'P019') & (led['branch_id'] == 'KOL001')].tail(1)['running_balance'].iloc[0])

MOV-ADJ-1519 P001 AHM001 quantity 38 balance before 4245 balance after 4207
  by hand subtracted table subtracted agree True
MOV-ADJ-524 P001 AHM001 quantity 33 balance before 13946 balance after 13979
  by hand added table added agree True
P019 KOL001 by hand: opening 92 in 95724 out 7135
  added 253 subtracted 81 net 172
  opening plus in minus out plus net 88853 last balance 88853
  table added 253 table subtracted 81
  table last balance 88853
